In [1]:
import pandas as pd
import numpy as np
import re

df = pd.read_csv('/content/customer_churn_raw_1100_rows.csv')

df.head()

,Customer_ID,Customer_Name,Gender,Age,City,State,Signup_Date,Last_Login,Tenure,Contract_Type,...,Discount,Satisfaction_Score,Customer_Service_Rating,Device_Type,Referral_Source,Churn,Churn_Date,Churn_Reason,Customer_Notes,Data_Source
0,cus_0249,ananya rao,M,38 years,warangal,telangana,Jul 11 2026,08/27/2026,1 months,M2M,...,Customer receives 25% discount,2 out of 5,Poor,Mobile,friend referral,Active,NaN,Not churned,NaN,CRM
1,cus_0106,Riya Kapoor,Male,49,Tirupati,Andhra Pradesh,Aug 26 2024,08/22/2026,24,Two-Year,...,No Discount,Average,4 stars,android phone,Friend,0,NaN,Not churned,customer happy with service,mobile app
2,cus_0197,SURESH BABU,F,26,NaN,Delhi,2026-07-21,2026-08-27,1,Monthly,...,0.10,5,Excellent,Android Phone,Google,0,NaN,NaN,moved to competitor,Mobile App
3,CUS-0891,Sneha Patel,Female,55,delhi,Not Available,16-Apr-26,30-Aug-26,1 years,Monthly,...,5%,Very Satisfied,Not Available,iPhone,Google,N,NaN,NaN,NaN,API
4,CUS-0333,amit shah,M,39,Not Available,Telangana,Jul 14 2026,08/27/2026,1 years,Monthly,...,15%,3 out of 5,2,Android,Google,No,NaN,NaN,customer happy with service,Import


In [2]:
df.shape

(1100, 30)

In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1100 entries, 0 to 1099
Data columns (total 30 columns):
 #   Column                   Non-Null Count  Dtype 
---  ------                   --------------  ----- 
 0   Customer_ID              1100 non-null   object
 1   Customer_Name            1100 non-null   object
 2   Gender                   1060 non-null   object
 3   Age                      1053 non-null   object
 4   City                     978 non-null    object
 5   State                    1012 non-null   object
 6   Signup_Date              1079 non-null   object
 7   Last_Login               820 non-null    object
 8   Tenure                   1068 non-null   object
 9   Contract_Type            1085 non-null   object
 10  Subscription_Plan        1074 non-null   object
 11  Monthly_Charges          1064 non-null   object
 12  Total_Charges            1054 non-null   object
 13  Payment_Method           1075 non-null   object
 14  Auto_Payment             1065 non-null  

In [4]:
df.isnull().sum()

,0
Customer_ID,0
Customer_Name,0
Gender,40
Age,47
City,122
State,88
Signup_Date,21
Last_Login,280
Tenure,32
Contract_Type,15


In [5]:
df.duplicated().sum()

np.int64(15)

## 1. Remove Customer ID

In [6]:
df.drop('Customer_ID', axis=1, inplace=True)

df.head()

,Customer_Name,Gender,Age,City,State,Signup_Date,Last_Login,Tenure,Contract_Type,Subscription_Plan,...,Discount,Satisfaction_Score,Customer_Service_Rating,Device_Type,Referral_Source,Churn,Churn_Date,Churn_Reason,Customer_Notes,Data_Source
0,ananya rao,M,38 years,warangal,telangana,Jul 11 2026,08/27/2026,1 months,M2M,Basic,...,Customer receives 25% discount,2 out of 5,Poor,Mobile,friend referral,Active,NaN,Not churned,NaN,CRM
1,Riya Kapoor,Male,49,Tirupati,Andhra Pradesh,Aug 26 2024,08/22/2026,24,Two-Year,basic,...,No Discount,Average,4 stars,android phone,Friend,0,NaN,Not churned,customer happy with service,mobile app
2,SURESH BABU,F,26,NaN,Delhi,2026-07-21,2026-08-27,1,Monthly,Basic,...,0.10,5,Excellent,Android Phone,Google,0,NaN,NaN,moved to competitor,Mobile App
3,Sneha Patel,Female,55,delhi,Not Available,16-Apr-26,30-Aug-26,1 years,Monthly,pro,...,5%,Very Satisfied,Not Available,iPhone,Google,N,NaN,NaN,NaN,API
4,amit shah,M,39,Not Available,Telangana,Jul 14 2026,08/27/2026,1 years,Monthly,PREMIUM,...,15%,3 out of 5,2,Android,Google,No,NaN,NaN,customer happy with service,Import


## 2. Common missing-value normalization

In [7]:
def normalize_missing(value):
    if pd.isna(value):
        return np.nan

    text = str(value).strip().lower()

    missing_values = ['', '-', 'missing', 'n/a', 'na', 'nan',
                      'none', 'not available', 'unknown', 'not disclosed']

    if text in missing_values:
        return np.nan

    return text

## 3. Clean individual columns

In [8]:
# Customer Name
df['Customer_Name'] = df['Customer_Name'].astype('string').str.strip().str.lower()
df['Customer_Name'] = df['Customer_Name'].apply(normalize_missing)

# Gender
df['Gender'] = df['Gender'].apply(normalize_missing)
df['Gender'] = df['Gender'].replace({'m': 'male', 'f': 'female'})
df['Gender'] = df['Gender'].fillna(df['Gender'].mode()[0])

# Age
df['Age'] = df['Age'].apply(normalize_missing)
df['Age'] = pd.to_numeric(df['Age'].str.extract(r'(\d+)', expand=False), errors='coerce')
df['Age'] = df['Age'].fillna(df['Age'].median()).astype(int)

# City
df['City'] = df['City'].apply(normalize_missing)
df['City'] = df['City'].replace({'new delhi': 'delhi'})
df['City'] = df['City'].fillna('unknown')

# State
df['State'] = df['State'].apply(normalize_missing)
df['State'] = df['State'].replace({
    'ap': 'andhra pradesh', 'ts': 'telangana', 'tn': 'tamil nadu',
    'kl': 'kerala', 'mh': 'maharashtra', 'gj': 'gujarat',
    'ka': 'karnataka'
})
df['State'] = df['State'].fillna('unknown')

In [9]:
df.head()

,Customer_Name,Gender,Age,City,State,Signup_Date,Last_Login,Tenure,Contract_Type,Subscription_Plan,...,Discount,Satisfaction_Score,Customer_Service_Rating,Device_Type,Referral_Source,Churn,Churn_Date,Churn_Reason,Customer_Notes,Data_Source
0,ananya rao,male,38,warangal,telangana,Jul 11 2026,08/27/2026,1 months,M2M,Basic,...,Customer receives 25% discount,2 out of 5,Poor,Mobile,friend referral,Active,NaN,Not churned,NaN,CRM
1,riya kapoor,male,49,tirupati,andhra pradesh,Aug 26 2024,08/22/2026,24,Two-Year,basic,...,No Discount,Average,4 stars,android phone,Friend,0,NaN,Not churned,customer happy with service,mobile app
2,suresh babu,female,26,unknown,delhi,2026-07-21,2026-08-27,1,Monthly,Basic,...,0.10,5,Excellent,Android Phone,Google,0,NaN,NaN,moved to competitor,Mobile App
3,sneha patel,female,55,delhi,unknown,16-Apr-26,30-Aug-26,1 years,Monthly,pro,...,5%,Very Satisfied,Not Available,iPhone,Google,N,NaN,NaN,NaN,API
4,amit shah,male,39,unknown,telangana,Jul 14 2026,08/27/2026,1 years,Monthly,PREMIUM,...,15%,3 out of 5,2,Android,Google,No,NaN,NaN,customer happy with service,Import


In [10]:
# Signup Date
df['Signup_Date'] = df['Signup_Date'].apply(normalize_missing)
df['Signup_Date'] = pd.to_datetime(df['Signup_Date'], errors='coerce', format='mixed', dayfirst=True)

# Last Login
df['Last_Login'] = df['Last_Login'].apply(normalize_missing)
df['Last_Login'] = pd.to_datetime(df['Last_Login'], errors='coerce', format='mixed', dayfirst=True)

# Tenure: convert years to months and retain month values
tenure_text = df['Tenure'].apply(normalize_missing).astype('string')
tenure_number = pd.to_numeric(tenure_text.str.extract(r'(\d+(?:\.\d+)?)', expand=False), errors='coerce')
df['Tenure'] = np.where(tenure_text.str.contains('year', na=False), tenure_number * 12, tenure_number)
df['Tenure'] = pd.to_numeric(df['Tenure'], errors='coerce')
df['Tenure'] = df['Tenure'].fillna(df['Tenure'].median()).round().astype(int)

# Contract Type
df['Contract_Type'] = df['Contract_Type'].apply(normalize_missing)
df['Contract_Type'] = df['Contract_Type'].replace({
    'm2m': 'monthly', 'month-to-month': 'monthly',
    'yearly': 'annual', 'two year': 'two-year'
})
df['Contract_Type'] = df['Contract_Type'].fillna(df['Contract_Type'].mode()[0])

# Subscription Plan
df['Subscription_Plan'] = df['Subscription_Plan'].apply(normalize_missing)
df['Subscription_Plan'] = df['Subscription_Plan'].str.replace(' plan', '', regex=False)
df['Subscription_Plan'] = df['Subscription_Plan'].fillna(df['Subscription_Plan'].mode()[0])

In [11]:
# Monthly Charges: values ending in K are multiplied by 1,000
monthly_text = df['Monthly_Charges'].apply(normalize_missing).astype('string')
monthly_number = pd.to_numeric(monthly_text.str.extract(r'(\d+(?:\.\d+)?)', expand=False), errors='coerce')
df['Monthly_Charges'] = np.where(monthly_text.str.contains('k', na=False), monthly_number * 1000, monthly_number)
df['Monthly_Charges'] = pd.to_numeric(df['Monthly_Charges'], errors='coerce')
df['Monthly_Charges'] = df['Monthly_Charges'].fillna(df['Monthly_Charges'].median())

# Total Charges: values ending in K are multiplied by 1,000
total_text = df['Total_Charges'].apply(normalize_missing).astype('string')
total_number = pd.to_numeric(total_text.str.extract(r'(\d+(?:\.\d+)?)', expand=False), errors='coerce')
df['Total_Charges'] = np.where(total_text.str.contains('k', na=False), total_number * 1000, total_number)
df['Total_Charges'] = pd.to_numeric(df['Total_Charges'], errors='coerce')
df['Total_Charges'] = df['Total_Charges'].fillna(df['Total_Charges'].median())

# Payment Method
df['Payment_Method'] = df['Payment_Method'].apply(normalize_missing)
df['Payment_Method'] = df['Payment_Method'].replace({
    'debit_card': 'debit card', 'creditcard': 'credit card',
    'cc': 'credit card'
})
df['Payment_Method'] = df['Payment_Method'].fillna(df['Payment_Method'].mode()[0])

# Auto Payment
df['Auto_Payment'] = df['Auto_Payment'].apply(normalize_missing)
df['Auto_Payment'] = df['Auto_Payment'].replace({
    'y': 'yes', '1': 'yes', 'n': 'no', '0': 'no'
})
df['Auto_Payment'] = df['Auto_Payment'].fillna(df['Auto_Payment'].mode()[0])

In [12]:
# Support Calls
support_text = df['Support_Calls'].apply(normalize_missing).astype('string')
df['Support_Calls'] = pd.to_numeric(support_text.str.extract(r'(\d+)', expand=False), errors='coerce')
df['Support_Calls'] = df['Support_Calls'].fillna(df['Support_Calls'].median()).astype(int)

# Complaints
complaint_text = df['Complaints'].apply(normalize_missing).astype('string')
df['Complaints'] = pd.to_numeric(complaint_text.str.extract(r'(\d+)', expand=False), errors='coerce')
df['Complaints'] = df['Complaints'].fillna(df['Complaints'].median()).astype(int)

# Internet Usage in GB
internet_text = df['Internet_Usage_GB'].apply(normalize_missing).astype('string')
df['Internet_Usage_GB'] = pd.to_numeric(internet_text.str.extract(r'(\d+(?:\.\d+)?)', expand=False), errors='coerce')
df['Internet_Usage_GB'] = df['Internet_Usage_GB'].fillna(df['Internet_Usage_GB'].median())

# Average Session Minutes: convert hours to minutes
session_text = df['Avg_Session_Minutes'].apply(normalize_missing).astype('string')
session_number = pd.to_numeric(session_text.str.extract(r'(\d+(?:\.\d+)?)', expand=False), errors='coerce')
df['Avg_Session_Minutes'] = np.where(session_text.str.contains('hour', na=False), session_number * 60, session_number)
df['Avg_Session_Minutes'] = pd.to_numeric(df['Avg_Session_Minutes'], errors='coerce')
df['Avg_Session_Minutes'] = df['Avg_Session_Minutes'].fillna(df['Avg_Session_Minutes'].median())

In [13]:
df.head()

,Customer_Name,Gender,Age,City,State,Signup_Date,Last_Login,Tenure,Contract_Type,Subscription_Plan,...,Discount,Satisfaction_Score,Customer_Service_Rating,Device_Type,Referral_Source,Churn,Churn_Date,Churn_Reason,Customer_Notes,Data_Source
0,ananya rao,male,38,warangal,telangana,2026-07-11,2026-08-27,1,monthly,basic,...,Customer receives 25% discount,2 out of 5,Poor,Mobile,friend referral,Active,NaN,Not churned,NaN,CRM
1,riya kapoor,male,49,tirupati,andhra pradesh,2024-08-26,2026-08-22,24,two-year,basic,...,No Discount,Average,4 stars,android phone,Friend,0,NaN,Not churned,customer happy with service,mobile app
2,suresh babu,female,26,unknown,delhi,2026-07-21,2026-08-27,1,monthly,basic,...,0.10,5,Excellent,Android Phone,Google,0,NaN,NaN,moved to competitor,Mobile App
3,sneha patel,female,55,delhi,unknown,2026-04-16,2026-08-30,12,monthly,pro,...,5%,Very Satisfied,Not Available,iPhone,Google,N,NaN,NaN,NaN,API
4,amit shah,male,39,unknown,telangana,2026-07-14,2026-08-27,12,monthly,premium,...,15%,3 out of 5,2,Android,Google,No,NaN,NaN,customer happy with service,Import


In [14]:
# Login Frequency
df['Login_Frequency'] = df['Login_Frequency'].apply(normalize_missing)
df['Login_Frequency'] = df['Login_Frequency'].replace({
    '2-3 times per week': '2-3 times/week'
})
df['Login_Frequency'] = df['Login_Frequency'].fillna(df['Login_Frequency'].mode()[0])

# Remove the discout
df = df.drop(['Discount'], axis = 1)

# Satisfaction Score: convert labels and x/5 formats to numeric score
satisfaction_text = df['Satisfaction_Score'].apply(normalize_missing)
df['Satisfaction_Score'] = satisfaction_text.replace({
    'very satisfied': '5', 'good': '4', 'average': '3', 'poor': '2'
})
df['Satisfaction_Score'] = pd.to_numeric(
    df['Satisfaction_Score'].astype('string').str.extract(r'(\d+)', expand=False),
    errors='coerce'
)
df['Satisfaction_Score'] = df['Satisfaction_Score'].fillna(df['Satisfaction_Score'].median()).round().astype(int)

# Customer Service Rating
rating_text = df['Customer_Service_Rating'].apply(normalize_missing)
df['Customer_Service_Rating'] = rating_text.replace({
    'excellent': '5', 'good': '4', 'average': '3', 'poor': '2'
})
df['Customer_Service_Rating'] = pd.to_numeric(
    df['Customer_Service_Rating'].astype('string').str.extract(r'(\d+)', expand=False),
    errors='coerce'
)
df['Customer_Service_Rating'] = df['Customer_Service_Rating'].fillna(
    df['Customer_Service_Rating'].median()
).round().astype(int)

In [15]:
# Device Type
df['Device_Type'] = df['Device_Type'].apply(normalize_missing)
df['Device_Type'] = df['Device_Type'].replace({
    'ios': 'iphone', 'android': 'android phone'
})
df['Device_Type'] = df['Device_Type'].fillna(df['Device_Type'].mode()[0])

# Referral Source
df['Referral_Source'] = df['Referral_Source'].apply(normalize_missing)
df['Referral_Source'] = df['Referral_Source'].replace({
    'friend referral': 'friend', 'google search': 'google'
})
df['Referral_Source'] = df['Referral_Source'].fillna(df['Referral_Source'].mode()[0])

# Churn: standardize target as yes/no
df['Churn'] = df['Churn'].astype('string').str.strip().str.lower()
df['Churn'] = df['Churn'].replace({
    '0': 'no', 'active': 'no', 'n': 'no',
    '1': 'yes', 'y': 'yes', 'churned': 'yes'
})

# Churn Date
df['Churn_Date'] = df['Churn_Date'].apply(normalize_missing)
df['Churn_Date'] = pd.to_datetime(df['Churn_Date'], errors='coerce', format='mixed', dayfirst=True)

# Churn Reason
df['Churn_Reason'] = df['Churn_Reason'].apply(normalize_missing)
df.loc[df['Churn'] == 'no', 'Churn_Reason'] = 'not churned'
df['Churn_Reason'] = df['Churn_Reason'].fillna('not specified')

In [16]:
# Customer Notes
df['Customer_Notes'] = df['Customer_Notes'].apply(normalize_missing)
df['Customer_Notes'] = df['Customer_Notes'].str.replace(r'!+', '', regex=True).str.strip()
df['Customer_Notes'] = df['Customer_Notes'].fillna('no notes')

# Data Source
df['Data_Source'] = df['Data_Source'].astype('string').str.strip().str.lower()
df['Data_Source'] = df['Data_Source'].replace({
    'crm_import': 'crm', 'import': 'crm'
})
df['Data_Source'] = df['Data_Source'].fillna(df['Data_Source'].mode()[0])

In [17]:
#removing the Churn Date
df = df.drop(['Churn_Date'], axis = 1)

## 4. Final data type and missing-value checks

In [18]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1100 entries, 0 to 1099
Data columns (total 27 columns):
 #   Column                   Non-Null Count  Dtype         
---  ------                   --------------  -----         
 0   Customer_Name            1100 non-null   object        
 1   Gender                   1100 non-null   object        
 2   Age                      1100 non-null   int64         
 3   City                     1100 non-null   object        
 4   State                    1100 non-null   object        
 5   Signup_Date              1067 non-null   datetime64[ns]
 6   Last_Login               820 non-null    datetime64[ns]
 7   Tenure                   1100 non-null   int64         
 8   Contract_Type            1100 non-null   object        
 9   Subscription_Plan        1100 non-null   object        
 10  Monthly_Charges          1100 non-null   float64       
 11  Total_Charges            1100 non-null   float64       
 12  Payment_Method           1100 non-

In [19]:
df['Last_Login'] = df['Last_Login'].fillna(df['Last_Login'].mode()[0])

In [20]:
df['Signup_Date'] = df['Signup_Date'].fillna(df['Last_Login'].mode()[0])

In [21]:
df.isnull().sum()

,0
Customer_Name,0
Gender,0
Age,0
City,0
State,0
Signup_Date,0
Last_Login,0
Tenure,0
Contract_Type,0
Subscription_Plan,0


In [22]:
df.duplicated().sum()

np.int64(22)

In [23]:
df = df.drop(['Churn_Reason','Customer_Notes','Data_Source'], axis = 1)

In [24]:
df.head(10)

,Customer_Name,Gender,Age,City,State,Signup_Date,Last_Login,Tenure,Contract_Type,Subscription_Plan,...,Support_Calls,Complaints,Internet_Usage_GB,Avg_Session_Minutes,Login_Frequency,Satisfaction_Score,Customer_Service_Rating,Device_Type,Referral_Source,Churn
0,ananya rao,male,38,warangal,telangana,2026-07-11,2026-08-27,1,monthly,basic,...,3,3,13.9,44.3,monthly,2,2,mobile,friend,no
1,riya kapoor,male,49,tirupati,andhra pradesh,2024-08-26,2026-08-22,24,two-year,basic,...,1,1,23.0,36.0,daily,3,4,android phone,friend,no
2,suresh babu,female,26,unknown,delhi,2026-07-21,2026-08-27,1,monthly,basic,...,6,0,21.1,87.0,daily,5,5,android phone,google,no
3,sneha patel,female,55,delhi,unknown,2026-04-16,2026-08-30,12,monthly,pro,...,2,1,18.4,12.0,2-3 times/week,5,3,iphone,google,no
4,amit shah,male,39,unknown,telangana,2026-07-14,2026-08-27,12,monthly,premium,...,0,1,21.7,75.0,weekly,3,2,android phone,google,no
5,meera nair,male,18,baroda,gujarat,2026-08-27,2026-08-17,10,monthly,premium,...,2,0,10.7,59.0,daily,4,4,android phone,friend,no
6,ananya rao,male,23,coimbatore,tamil nadu,2026-07-13,2026-08-27,12,monthly,pro,...,1,2,9.5,54.0,monthly,3,2,mobile,friend,no
7,harsha vardhan,female,24,tirupati,andhra pradesh,2023-06-09,2026-08-24,39,monthly,basic,...,5,1,10.2,62.0,weekly,4,3,laptop,friend,no
8,rahul sharma,male,51,vijayawada,andhra pradesh,2025-08-06,2026-08-25,13,monthly,premium,...,4,1,28.7,28.0,weekly,4,3,android phone,advertisement,no
9,pooja nair,female,44,baroda,gujarat,2026-06-22,2026-08-27,2,annual,premium,...,1,2,74.4,65.0,daily,5,5,iphone,social media,no


## 5. Export the cleaned dataset

In [25]:
df.to_csv('/content/customer_churn_cleaned.csv', index=False)

print('Cleaned dataset saved successfully!')
print('Final shape:', df.shape)

from google.colab import files
files.download('/content/customer_churn_cleaned.csv')

Cleaned dataset saved successfully!
Final shape: (1100, 24)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>